In [83]:
import numpy as np
import matplotlib.pyplot as plt # plotting data
import pandas as pd # since we have mixed data types, pandas will be a better option
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score


from sklearn.linear_model import Ridge, Lasso, LassoCV, ElasticNet, SGDRegressor

In [84]:
##############################################
# Data Preprocessing and Feature Engineering #
##############################################

In [85]:
# load the dataset
train_data = pd.read_csv("dataset/train.csv")
test_data = pd.read_csv("dataset/test.csv")

In [86]:
# Splitting data into features X and target y
X = train_data.drop(columns = ['SalePrice', 'Id', 'GarageArea', 'TotRmsAbvGrd']) # We drop SalePrice because it's our target, and drop Id because it has no meaning while training the model
y = train_data['SalePrice']
X_test = test_data.drop(columns = ['Id', 'GarageArea', 'TotRmsAbvGrd'])

In [87]:
# Splitting features into two categories, Categorical and numerical, to apply one-hot encoding to the categorical data
X_categorical = X.select_dtypes(include = ['object']).columns.tolist()
X_numerical = X.select_dtypes(exclude = ['object']).columns.tolist()

In [88]:
# Finding correlated data
X[X_numerical].corr()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageCars,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
MSSubClass,1.000000,-0.386347,-0.139781,0.032628,-0.059316,0.027850,0.040581,0.022936,-0.069836,-0.065649,...,-0.040110,-0.012579,-0.006100,-0.012037,-0.043825,-0.026030,0.008283,-0.007683,-0.013585,-0.021407
LotFrontage,-0.386347,1.000000,0.426095,0.251646,-0.059213,0.123349,0.088866,0.193458,0.233633,0.049900,...,0.285691,0.088521,0.151972,0.010700,0.070029,0.041383,0.206167,0.003368,0.011200,0.007450
LotArea,-0.139781,0.426095,1.000000,0.105806,-0.005636,0.014228,0.013788,0.104160,0.214103,0.111170,...,0.154871,0.171698,0.084774,-0.018340,0.020423,0.043160,0.077672,0.038068,0.001205,-0.014261
OverallQual,0.032628,0.251646,0.105806,1.000000,-0.091932,0.572323,0.550684,0.411876,0.239666,-0.059119,...,0.600671,0.238923,0.308819,-0.113937,0.030371,0.064886,0.065166,-0.031406,0.070815,-0.027347
OverallCond,-0.059316,-0.059213,-0.005636,-0.091932,1.000000,-0.375983,0.073741,-0.128101,-0.046231,0.040229,...,-0.185758,-0.003334,-0.032589,0.070356,0.025504,0.054811,-0.001985,0.068777,-0.003511,0.043950
YearBuilt,0.027850,0.123349,0.014228,0.572323,-0.375983,1.000000,0.592855,0.315707,0.249503,-0.049107,...,0.537850,0.224880,0.188686,-0.387268,0.031355,-0.050364,0.004950,-0.034383,0.012398,-0.013618
YearRemodAdd,0.040581,0.088866,0.013788,0.550684,0.073741,0.592855,1.000000,0.179618,0.128451,-0.067759,...,0.420622,0.205726,0.226298,-0.193919,0.045286,-0.038740,0.005829,-0.010286,0.021490,0.035743
MasVnrArea,0.022936,0.193458,0.104160,0.411876,-0.128101,0.315707,0.179618,1.000000,0.264736,-0.072319,...,0.364204,0.159718,0.125703,-0.110204,0.018796,0.061466,0.011723,-0.029815,-0.005965,-0.008201
BsmtFinSF1,-0.069836,0.233633,0.214103,0.239666,-0.046231,0.249503,0.128451,0.264736,1.000000,-0.050117,...,0.224054,0.204306,0.111761,-0.102303,0.026451,0.062021,0.140491,0.003571,-0.015727,0.014359
BsmtFinSF2,-0.065649,0.049900,0.111170,-0.059119,0.040229,-0.049107,-0.067759,-0.072319,-0.050117,1.000000,...,-0.038264,0.067898,0.003093,0.036543,-0.029993,0.088871,0.041709,0.004940,-0.015211,0.031706


In [89]:
df_imputed = X[X_numerical].apply(lambda x: x.fillna(x.mean()), axis=0)

# Compute correlation matrix
correlation_matrix = df_imputed.corr()

# Find the most correlated variables (excluding self-correlation)
correlation_pairs = correlation_matrix.unstack()
correlation_pairs = correlation_pairs[correlation_pairs != 1]  # Remove self-correlation
most_correlated = correlation_pairs.sort_values(ascending=False).drop_duplicates()

# Display the most correlated variables
print(most_correlated.head())

1stFlrSF      TotalBsmtSF    0.819530
YearBuilt     GarageYrBlt    0.780555
GrLivArea     2ndFlrSF       0.687501
BsmtFullBath  BsmtFinSF1     0.649212
GrLivArea     FullBath       0.630012
dtype: float64


In [90]:
X.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


In [91]:
# Filling the missing numerical values with the mean
#numerical_transformer = SimpleImputer(strategy = 'mean')
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())  #scaling for Lasso
])

In [56]:
# Filling the missing categorical values with the most frequent, and convert them to numerical using one-hot encoding
    # Pipeline encapsulates the process of data preparation and model training into a single object
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

In [57]:
# specify which features in the dataset are numerical and which are categorical
preprocessor = ColumnTransformer( transformers = [
    ('num', numerical_transformer, X_numerical),
    ('cat', categorical_transformer, X_categorical)
])

In [58]:
###############


In [93]:
# Define the model
#model = Pipeline(steps = [
#    ('preprocessor', preprocessor),
#    ('model', LinearRegression())
#])


model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lasso', Lasso(max_iter=500000))
])

In [94]:
param_grid = {
    'lasso__alpha': [0.01, 0.1, 1, 10, 100]  #different alpha values for tuning
}

In [95]:
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error')

In [96]:
# Split the data into training and validation (testing) sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [97]:
# Train the model
#model.fit(X_train, y_train)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         SimpleImputer(),
                                                                         ['MSSubClass',
                                                                          'LotFrontage',
                                                                          'LotArea',
                                                                          'OverallQual',
                                                                          'OverallCond',
                                                                          'YearBuilt',
                                                                          'YearRemodAdd',
                                                                          'MasVnrArea',
                                                                          'BsmtFinSF1',
                                                                          'BsmtFinSF2',
                                                                          'BsmtUnfSF',
                                                                          'TotalBsmtSF',
                                                                          '1stFlrSF',
                                                                          '2ndFlrSF',
                                                                          'LowQualFinSF',
                                                                          'GrLivArea',
                                                                          'BsmtFullBath',
                                                                          'BsmtHalfBath...
                                                                          'HouseStyle',
                                                                          'RoofStyle',
                                                                          'RoofMatl',
                                                                          'Exterior1st',
                                                                          'Exterior2nd',
                                                                          'MasVnrType',
                                                                          'ExterQual',
                                                                          'ExterCond',
                                                                          'Foundation',
                                                                          'BsmtQual',
                                                                          'BsmtCond',
                                                                          'BsmtExposure',
                                                                          'BsmtFinType1',
                                                                          'BsmtFinType2',
                                                                          'Heating',
                                                                          'HeatingQC',
                                                                          'CentralAir',
                                                                          'Electrical', ...])])),
                                       ('lasso', Lasso(max_iter=500000))]),
             param_grid={'lasso__alpha': [0.01, 0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

In [98]:
# Make predictions
y_pred = grid_search.predict(X_val)

In [99]:
# Predict test file
y_test = grid_search.predict(X_test)

In [100]:
# saving it into a new file called LinearRegressionResult
output = pd.DataFrame({'Id': test_data['Id'], 'SalePrice': y_test})
output.to_csv('LinearRegressionResult.csv', index=False)

In [101]:
# Evaluate the model
mse = mean_squared_error(y_val, y_pred)
print(f'Mean Squared Error: {mse}')

Mean Squared Error: 793478724.9057013


In [102]:
rmse = mean_squared_error(y_val, y_pred, squared = False)
print(rmse)

28168.754408132805


In [103]:
r2_score(y_val, y_pred)

0.8965521539146264

In [ ]:
cv_scores = cross_val_score(grid_search, X, y, cv=5, scoring='neg_mean_squared_error')
print(f'Cross-Validated MSE: {-np.mean(cv_scores)}')

In [ ]:
# Plotting the results
plt.figure(figsize=(10, 6))
plt.bar(output['Id'][:50], output['SalePrice'][:50])
plt.xlabel('Id')
plt.ylabel('SalePrice')
plt.title('House Prices Predictions')
plt.xticks(rotation=90)
plt.show()